# Aether Stage 2 — clean-audio inference

Five-cell inference notebook for an RTX 3060 or larger NVIDIA GPU.

1. Clone the `stage2` branch and install dependencies.
2. Authenticate and download every required checkpoint.
3. Construct Mimi + AetherSpeech + Connector + 4-bit Qwen3-4B.
4. Set `AUDIO_PATH`, run the audio front end, and inspect the exact Connector embeddings passed to Qwen.
5. Run Qwen generation, producing both a diagnostic transcript and an experimental direct answer.

The audio should contain one clean spoken question. WAV, FLAC and other formats supported by `soundfile` are accepted.

In [ ]:
# CELL 1 — clone Stage 2 and install every dependency
import logging
import os
import subprocess
import sys
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)

REPO_URL = "https://github.com/karl4th/aether-v3.git"
if Path("/content").exists():
    REPO_DIR = Path("/content/aether-v3")
else:
    REPO_DIR = Path.home() / "aether-v3-runtime"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "stage2"], check=True)
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "stage2",
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "stage2"], check=True)
subprocess.run(
    ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/stage2"],
    check=True,
)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        str(REPO_DIR),
        "bitsandbytes",
        "accelerate",
        "huggingface_hub",
    ],
    check=True,
)

for module_name in list(sys.modules):
    if module_name == "aether_v3" or module_name.startswith("aether_v3."):
        del sys.modules[module_name]

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
print("CELL 1 COMPLETE:", {"repo": str(REPO_DIR), "commit": GIT_COMMIT}, flush=True)

In [ ]:
# CELL 2 — authenticate and download all checkpoints
import getpass

from huggingface_hub import hf_hub_download

from aether_v3.config import load_config

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if "HF_TOKEN" not in os.environ:
    colab_token = userdata.get("HF_TOKEN") if userdata is not None else None
    os.environ["HF_TOKEN"] = colab_token or getpass.getpass("Hugging Face read token: ")
assert os.environ["HF_TOKEN"], "A Hugging Face read token is required"

CONFIG_PATH = REPO_DIR / "configs/stage2_r1_full.yaml"
cfg = load_config(CONFIG_PATH)

STAGE1_REPO = "manifestro/aetherASR-EN-v0.1"
STAGE1_FILE = "last.pt"
STAGE2_REPO = "manifestro/aether"
STAGE2_FILE = "stage2/best_wer.pt"

stage1_path = hf_hub_download(
    repo_id=STAGE1_REPO,
    filename=STAGE1_FILE,
    revision=cfg.stage2_train.stage1_revision,
    token=os.environ["HF_TOKEN"],
)
stage2_path = hf_hub_download(
    repo_id=STAGE2_REPO,
    filename=STAGE2_FILE,
    token=os.environ["HF_TOKEN"],
)

assert Path(stage1_path).is_file()
assert Path(stage2_path).is_file()
print(
    "CELL 2 COMPLETE:",
    {"stage1": stage1_path, "stage2": stage2_path, "qwen": cfg.llm.model_id},
    flush=True,
)

In [ ]:
# CELL 3 — construct Mimi + AetherSpeech + Connector + 4-bit Qwen
import dataclasses
import gc
import json
import time

import numpy as np
import soundfile as sf
import torch
import torchaudio
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from aether_v3.models.aether_speech_llm import AetherSpeechLLM
from aether_v3.models.mimi_wrapper import FrozenMimi
from aether_v3.training.stage2_utils import load_stage1_encoder
from aether_v3.training.train_stage2 import load_stage2_trainable_weights

assert torch.cuda.is_available(), "An NVIDIA CUDA GPU is required"
device = torch.device("cuda")

def cuda_sync():
    torch.cuda.synchronize(device)

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(
    cfg.llm.model_id,
    revision=cfg.llm.revision,
    token=os.environ["HF_TOKEN"],
)
print("QWEN: loading 4-bit weights", flush=True)
llm = AutoModelForCausalLM.from_pretrained(
    cfg.llm.model_id,
    revision=cfg.llm.revision,
    token=os.environ["HF_TOKEN"],
    quantization_config=quantization,
    device_map={"": 0},
    dtype=torch.float16,
    attn_implementation="sdpa",
)

inference_llm_cfg = dataclasses.replace(
    cfg.llm,
    dtype="float16",
    gradient_checkpointing=False,
)
model = AetherSpeechLLM(
    cfg.aether_speech,
    cfg.connector,
    inference_llm_cfg,
    speech_frozen=True,
    llm=llm,
)
stage1_checkpoint = load_stage1_encoder(stage1_path, model.encoder)
model.encoder.to(device)
model.connector.to(device=device, dtype=torch.float16)
stage2_checkpoint = load_stage2_trainable_weights(model, stage2_path, device)
model.eval()

# Mimi starts on CPU and moves to CUDA only while encoding the selected file.
mimi = FrozenMimi(
    cfg.mimi.pretrained_id,
    cfg.mimi.num_quantizers,
    device="cpu",
)
cuda_sync()
MODEL_LOAD_SECONDS = time.perf_counter() - model_load_started

assert model.connector.bridge.output_scale.dtype == torch.float32
MODEL_INFO = {
    "git_commit": GIT_COMMIT,
    "gpu": torch.cuda.get_device_name(0),
    "vram_total_gb": torch.cuda.get_device_properties(0).total_memory / 2**30,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "cuda_version": torch.version.cuda,
    "qwen_model": cfg.llm.model_id,
    "qwen_quantization": "NF4 4-bit, double quantization, FP16 compute",
    "stage1_step": stage1_checkpoint.get("step"),
    "stage2_step": stage2_checkpoint.get("step"),
    "output_scale": float(model.connector.bridge.output_scale.detach()),
    "model_load_seconds": MODEL_LOAD_SECONDS,
    "allocated_after_load_gb": torch.cuda.memory_allocated() / 2**30,
    "reserved_after_load_gb": torch.cuda.memory_reserved() / 2**30,
}
print("CELL 3 COMPLETE:\n", json.dumps(MODEL_INFO, indent=2), sep="", flush=True)

In [ ]:
# CELL 4 — run audio -> Mimi -> AetherSpeech -> Connector and inspect Qwen input
import hashlib
import statistics

from aether_v3.eval.metrics import compute_cer, compute_wer

AUDIO_PATH = Path("/content/question.wav")  # <-- change this path
REFERENCE_TRANSCRIPT = ""  # optional exact text for WER/CER
BENCHMARK_WARM_RUNS = 3
MAX_TRANSCRIPT_TOKENS = 256
MAX_ANSWER_TOKENS = 96

assert AUDIO_PATH.is_file(), f"Audio file not found: {AUDIO_PATH}"
assert BENCHMARK_WARM_RUNS >= 1

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def percentile(values, fraction):
    ordered = sorted(values)
    index = min(len(ordered) - 1, max(0, round((len(ordered) - 1) * fraction)))
    return ordered[index]

def timed_cuda(callable_):
    cuda_sync()
    started = time.perf_counter()
    value = callable_()
    cuda_sync()
    return value, time.perf_counter() - started

audio_started = time.perf_counter()
waveform, sample_rate = sf.read(AUDIO_PATH, dtype="float32", always_2d=True)
waveform = waveform.mean(axis=1)
assert waveform.size > 0, "Audio is empty"
assert np.isfinite(waveform).all(), "Audio contains NaN or Inf"

if sample_rate != cfg.mimi.sampling_rate:
    waveform = torchaudio.functional.resample(
        torch.from_numpy(waveform),
        sample_rate,
        cfg.mimi.sampling_rate,
    ).numpy()
    sample_rate = cfg.mimi.sampling_rate

audio_prepare_seconds = time.perf_counter() - audio_started
duration = len(waveform) / sample_rate
peak = float(np.abs(waveform).max(initial=0.0))
rms = float(np.sqrt(np.mean(np.square(waveform))))
assert duration >= 0.25, f"Audio is too short: {duration:.2f}s"
assert peak > 1e-4, f"Audio appears silent: peak={peak}"

torch.cuda.reset_peak_memory_stats()
allocated_before_gb = torch.cuda.memory_allocated() / 2**30
reserved_before_gb = torch.cuda.memory_reserved() / 2**30

print("MIMI: encoding", flush=True)
cuda_sync()
mimi_transfer_started = time.perf_counter()
mimi.model.to(device)
mimi.device = device
cuda_sync()
mimi_transfer_to_gpu_seconds = time.perf_counter() - mimi_transfer_started
semantic_codes, mimi_seconds = timed_cuda(lambda: mimi.encode_semantic([waveform])[0])
cuda_sync()
mimi_release_started = time.perf_counter()
mimi.model.to("cpu")
mimi.device = torch.device("cpu")
gc.collect()
torch.cuda.empty_cache()
cuda_sync()
mimi_release_seconds = time.perf_counter() - mimi_release_started

codes = torch.from_numpy(semantic_codes).to(device=device, dtype=torch.long).unsqueeze(0)
speech_mask = torch.ones_like(codes, dtype=torch.bool)
speech_states, aetherspeech_seconds = timed_cuda(
    lambda: model.encoder(codes, speech_mask)
)
assert speech_states.shape == (1, len(semantic_codes), cfg.aether_speech.hidden_size)


# Connector produces continuous vectors, not text tokens. This cell prints the
# exact tensor statistics and a nearest-token projection for interpretation.
@torch.inference_mode()
def tensor_stats(values):
    original_dtype = str(values.dtype)
    values = values.detach().float()
    row_norms = values.norm(dim=-1)
    return {
        "shape": list(values.shape),
        "dtype": original_dtype,
        "mean": float(values.mean()),
        "std": float(values.std()),
        "rms": float(values.square().mean().sqrt()),
        "l2_p50": float(row_norms.quantile(0.50)),
        "l2_p90": float(row_norms.quantile(0.90)),
        "l2_p99": float(row_norms.quantile(0.99)),
        "finite": bool(torch.isfinite(values).all()),
    }

connector_dtype = next(model.connector.parameters()).dtype
with torch.inference_mode():
    speech_embeds, connector_mask = model.connector(
        speech_states.to(connector_dtype),
        speech_mask,
    )

inspection_prompt = "Transcribe the following speech exactly. Output only the transcript:\n"
inspection_prefix = tokenizer(
    inspection_prompt,
    add_special_tokens=False,
    return_tensors="pt",
)
empty_ids = torch.empty(1, 0, dtype=torch.long, device=device)
empty_mask = torch.empty(1, 0, dtype=torch.bool, device=device)
with torch.inference_mode():
    qwen_input = model.build_inputs_embeds(
        inspection_prefix["input_ids"].to(device),
        inspection_prefix["attention_mask"].to(device=device, dtype=torch.bool),
        speech_embeds,
        connector_mask,
        empty_ids,
        empty_mask,
    )

prefix_length = int(inspection_prefix["attention_mask"].sum())
speech_length = int(connector_mask.sum())
assert int(qwen_input.attention_mask.sum()) == prefix_length + 1 + speech_length + 1

# Project a small, evenly-spaced selection of speech vectors onto Qwen's token
# embedding table. These are nearest lexical directions, not tokens generated
# by the model and not a hidden transcript.
selected_count = min(24, speech_length)
selected_frames = torch.linspace(
    0,
    speech_length - 1,
    selected_count,
    device=device,
).round().long().unique()
selected_vectors = torch.nn.functional.normalize(
    speech_embeds[0, selected_frames].float(),
    dim=-1,
)
embedding_weight = model.llm.get_input_embeddings().weight
best_scores = torch.full((len(selected_frames), 3), -float("inf"), device=device)
best_ids = torch.full((len(selected_frames), 3), -1, dtype=torch.long, device=device)
chunk_size = 8192
with torch.inference_mode():
    for start in range(0, embedding_weight.shape[0], chunk_size):
        chunk = torch.nn.functional.normalize(
            embedding_weight[start : start + chunk_size].float(),
            dim=-1,
        )
        scores = selected_vectors @ chunk.T
        chunk_scores, chunk_ids = scores.topk(3, dim=-1)
        combined_scores = torch.cat([best_scores, chunk_scores], dim=-1)
        combined_ids = torch.cat([best_ids, chunk_ids + start], dim=-1)
        best_scores, positions = combined_scores.topk(3, dim=-1)
        best_ids = combined_ids.gather(1, positions)

nearest_token_projection = []
for row, frame_index in enumerate(selected_frames.tolist()):
    candidates = []
    for token_id, score in zip(best_ids[row].tolist(), best_scores[row].tolist(), strict=True):
        candidates.append(
            {
                "token_id": token_id,
                "token": tokenizer.convert_ids_to_tokens(token_id),
                "decoded": tokenizer.decode([token_id]),
                "cosine": score,
            }
        )
    nearest_token_projection.append(
        {
            "speech_frame": frame_index,
            "time_seconds_approx": frame_index / 12.5,
            "first_8_values": speech_embeds[0, frame_index, :8].float().cpu().tolist(),
            "nearest_qwen_tokens": candidates,
        }
    )

CONNECTOR_INPUT_DIAGNOSTIC = {
    "warning": (
        "Connector output is continuous. Nearest Qwen tokens are only a cosine "
        "projection for inspection, not a transcript or the tokens Qwen receives."
    ),
    "sequence_layout": "[text prefix] [SPEECH_START] [speech embeddings] [SPEECH_END]",
    "prefix_tokens": prefix_length,
    "speech_frames": speech_length,
    "total_qwen_input_positions": int(qwen_input.attention_mask.sum()),
    "speech_states_before_connector": tensor_stats(speech_states[0, :speech_length]),
    "connector_output": tensor_stats(speech_embeds[0, :speech_length]),
    "speech_start_embedding": tensor_stats(model.connector.speech_start.unsqueeze(0)),
    "speech_end_embedding": tensor_stats(model.connector.speech_end.unsqueeze(0)),
    "full_qwen_inputs_embeds": tensor_stats(
        qwen_input.inputs_embeds[0, qwen_input.attention_mask[0]]
    ),
    "nearest_token_projection": nearest_token_projection,
}

print(
    "\nCONNECTOR -> QWEN INPUT DIAGNOSTIC:\n",
    json.dumps(CONNECTOR_INPUT_DIAGNOSTIC, indent=2, ensure_ascii=False),
    sep="",
    flush=True,
)
connector_diagnostic_path = AUDIO_PATH.with_suffix(".connector_input.json")
connector_diagnostic_path.write_text(
    json.dumps(CONNECTOR_INPUT_DIAGNOSTIC, indent=2, ensure_ascii=False)
)
print("CONNECTOR DIAGNOSTIC SAVED:", connector_diagnostic_path, flush=True)


In [ ]:
# CELL 5 — run Connector -> Qwen generation and benchmark

def generate(prompt, max_new_tokens):
    prefix = tokenizer(prompt, add_special_tokens=False, return_tensors="pt")
    batch = {
        "speech_states": speech_states,
        "speech_mask": speech_mask,
        "prefix_ids": prefix["input_ids"].to(device),
        "prefix_mask": prefix["attention_mask"].to(device=device, dtype=torch.bool),
    }
    token_ids, elapsed = timed_cuda(
        lambda: model.generate_cached(
            batch,
            eos_token_id=tokenizer.eos_token_id,
            max_new_tokens=max_new_tokens,
            use_kv_cache=True,
        )[0]
    )
    text = tokenizer.decode(token_ids, skip_special_tokens=True).strip()
    return text, token_ids, elapsed

transcription_prompt = (
    "Transcribe the following speech exactly. Output only the transcript:\n"
)
answer_prompt = "Answer the spoken question directly. Give only a short answer:\n"

# First run represents the first request after model loading.
transcript, transcript_ids, transcript_first_seconds = generate(
    transcription_prompt,
    MAX_TRANSCRIPT_TOKENS,
)
answer, answer_ids, answer_first_seconds = generate(answer_prompt, MAX_ANSWER_TOKENS)

# Repeated deterministic runs measure warm latency without model loading.
transcript_warm_seconds = []
answer_warm_seconds = []
for benchmark_index in range(BENCHMARK_WARM_RUNS):
    print(f"BENCHMARK: warm run {benchmark_index + 1}/{BENCHMARK_WARM_RUNS}", flush=True)
    _, repeated_transcript_ids, elapsed = generate(
        transcription_prompt,
        MAX_TRANSCRIPT_TOKENS,
    )
    assert repeated_transcript_ids == transcript_ids
    transcript_warm_seconds.append(elapsed)
    _, repeated_answer_ids, elapsed = generate(answer_prompt, MAX_ANSWER_TOKENS)
    assert repeated_answer_ids == answer_ids
    answer_warm_seconds.append(elapsed)

transcript_tokens = len(transcript_ids)
answer_tokens = len(answer_ids)
transcript_warm_p50 = statistics.median(transcript_warm_seconds)
answer_warm_p50 = statistics.median(answer_warm_seconds)
front_end_seconds = (
    audio_prepare_seconds
    + mimi_transfer_to_gpu_seconds
    + mimi_seconds
    + mimi_release_seconds
    + aetherspeech_seconds
)
transcription_e2e_first = front_end_seconds + transcript_first_seconds
answer_e2e_first = front_end_seconds + answer_first_seconds

quality = None
if REFERENCE_TRANSCRIPT.strip():
    quality = {
        "reference": REFERENCE_TRANSCRIPT.strip(),
        "wer": compute_wer([REFERENCE_TRANSCRIPT], [transcript]),
        "cer": compute_cer([REFERENCE_TRANSCRIPT], [transcript]),
    }

result = {
    "model": MODEL_INFO,
    "connector_input": CONNECTOR_INPUT_DIAGNOSTIC,
    "audio": {
        "path": str(AUDIO_PATH),
        "sha256": file_sha256(AUDIO_PATH),
        "seconds": duration,
        "sample_rate": sample_rate,
        "peak": peak,
        "rms": rms,
        "semantic_frames": len(semantic_codes),
    },
    "outputs": {
        "transcript": transcript,
        "direct_answer_experimental": answer,
        "transcript_tokens": transcript_tokens,
        "answer_tokens": answer_tokens,
        "quality": quality,
    },
    "latency_seconds": {
        "audio_load_and_resample": audio_prepare_seconds,
        "mimi_transfer_to_gpu": mimi_transfer_to_gpu_seconds,
        "mimi_encode": mimi_seconds,
        "mimi_release_to_cpu": mimi_release_seconds,
        "aetherspeech": aetherspeech_seconds,
        "front_end_total": front_end_seconds,
        "transcription_first": transcript_first_seconds,
        "answer_first": answer_first_seconds,
        "transcription_end_to_end_first": transcription_e2e_first,
        "answer_end_to_end_first": answer_e2e_first,
        "transcription_warm_runs": transcript_warm_seconds,
        "answer_warm_runs": answer_warm_seconds,
        "transcription_warm_p50": transcript_warm_p50,
        "transcription_warm_p90": percentile(transcript_warm_seconds, 0.90),
        "answer_warm_p50": answer_warm_p50,
        "answer_warm_p90": percentile(answer_warm_seconds, 0.90),
    },
    "throughput": {
        "semantic_frames_per_second": len(semantic_codes) / max(mimi_seconds, 1e-9),
        "transcription_tokens_per_second_first": transcript_tokens
        / max(transcript_first_seconds, 1e-9),
        "answer_tokens_per_second_first": answer_tokens / max(answer_first_seconds, 1e-9),
        "transcription_real_time_factor_first": transcription_e2e_first / duration,
        "answer_real_time_factor_first": answer_e2e_first / duration,
    },
    "memory_gb": {
        "allocated_before": allocated_before_gb,
        "reserved_before": reserved_before_gb,
        "peak_allocated": torch.cuda.max_memory_allocated() / 2**30,
        "peak_reserved": torch.cuda.max_memory_reserved() / 2**30,
        "allocated_after": torch.cuda.memory_allocated() / 2**30,
        "reserved_after": torch.cuda.memory_reserved() / 2**30,
    },
}

print("\nTRANSCRIPT:\n", transcript, sep="", flush=True)
print("\nDIRECT ANSWER:\n", answer, sep="", flush=True)
print("\nBENCHMARK REPORT:\n", json.dumps(result, indent=2), sep="", flush=True)

result_path = AUDIO_PATH.with_suffix(".aether.json")
result_path.write_text(json.dumps(result, indent=2, ensure_ascii=False))
print("SAVED:", result_path, flush=True)